In [ ]:
import oracledb, os, json, paramiko, datetime
from google.cloud import secretmanager
import pandas as pd
import numpy as np
from io import StringIO

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def set_secrets_as_envs():
  secrets = secretmanager.SecretManagerServiceClient()
  resource_name = f"{os.environ['KNADA_TEAM_SECRET']}/versions/latest"
  secret = secrets.access_secret_version(name=resource_name)
  secret_str = secret.payload.data.decode('UTF-8')
  secrets = json.loads(secret_str)
  os.environ.update(secrets)

In [ ]:
def oracle_secrets():
  set_secrets_as_envs()
  return dict(
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host = os.getenv('DBT_ORCL_HOST'),
    service = os.getenv('DBT_ORCL_SERVICE'),
    sftp_username = os.getenv('SFTP_USERNAME'),
    sftpkey = os.getenv('SFTPKEY'),
    sftp_host = os.getenv('SFTP_HOST'),
    port = os.getenv('SFTP_PORT'),
    bidrag_filomraade = os.getenv('BIDRAG_FILOMRAADE'),
    encoding="UTF-8",
    nencoding="UTF-8"
  )

oracle_secrets = oracle_secrets()

In [ ]:
def get_periode(fmt="%Y%m"):
    today = datetime.date.today() 
    first = today.replace(day=1) 
    lastMonth = first - datetime.timedelta(days=1) 

    return lastMonth.strftime(fmt) 

In [ ]:
periode = get_periode("%Y-%m")
periode_yyyymm = get_periode()

In [ ]:
os.makedirs("data", exist_ok=True)

keyfile = StringIO(oracle_secrets['sftpkey'])
mykey = paramiko.RSAKey.from_private_key(keyfile, password=np.nan)

# Open a transport
host,port = oracle_secrets['sftp_host'], 22 
transport = paramiko.Transport((host,port))

# Auth    
username= oracle_secrets['sftp_username']
transport.connect(username=username,pkey=mykey)

stonader = ['BIDRAG_Bisys', 'BIDRAG18AAR_Bisys', 'OPPFOSTRINGSBIDRAG_Bisys', 'FORSKUDD_Bisys']

with paramiko.SFTPClient.from_transport(transport) as sftp:
    for s in stonader:
        sftp.get(f'./{oracle_secrets["bidrag_filomraade"]}/Avstemming_{periode}_{s}.txt', f'./data/Avstemming_{periode}_{s}.txt')

# Close
if sftp: sftp.close()
if transport: transport.close()

In [ ]:
df_bidrag = pd.read_csv(f'data/Avstemming_{periode}_BIDRAG_Bisys.txt', sep='\t', names=['SAKSNR', 'ANTALL_BARN', 'BELOP'] ,dtype={"SAKSNR": str})
df_bidrag['STONADSTYPE'] = 'BIDRAG'
df_bidrag['AAR_MAANED'] = periode_yyyymm

In [ ]:
df_18bidrag = pd.read_csv(f'data/Avstemming_{periode}_BIDRAG18AAR_Bisys.txt', sep='\t', names=['SAKSNR', 'ANTALL_BARN', 'BELOP'], dtype={"SAKSNR": str})
df_18bidrag['STONADSTYPE'] = 'BIDRAG18AAR'
df_18bidrag['AAR_MAANED'] = periode_yyyymm

In [ ]:
df_oppfostering = pd.read_csv(f'data/Avstemming_{periode}_OPPFOSTRINGSBIDRAG_Bisys.txt', sep='\t', names=['SAKSNR', 'ANTALL_BARN', 'BELOP'], dtype={"SAKSNR": str})
df_oppfostering['STONADSTYPE'] = 'OPPFOSTRINGSBIDRAG'
df_oppfostering['AAR_MAANED'] = periode_yyyymm

In [ ]:
df_combined = pd.concat([df_bidrag, df_18bidrag, df_oppfostering], ignore_index=True)

In [ ]:
df_forskudd = pd.read_csv(f'data/Avstemming_{periode}_FORSKUDD_Bisys.txt', sep='\t', names=['SAKSNR', 'ANTALL_BARN', 'BELOP'] ,dtype={"SAKSNR": str})
df_forskudd['AAR_MAANED'] = periode_yyyymm

In [ ]:
user = oracle_secrets['user'] + '[DVH_FAM_BB]'
dsn_tns = oracledb.makedsn(oracle_secrets['host'], 1521, service_name = oracle_secrets['service'])

with oracledb.connect(user=user, password = oracle_secrets['password'], dsn=dsn_tns) as conn:
    with conn.cursor() as cursor:
        rows = [tuple(x) for x in df_combined.values]
        cursor.executemany('''INSERT INTO BB_BIDRAG_FRA_KILDE (SAKSNR, ANTALL_BARN, BELOP, STONADSTYPE, AAR_MAANED) 
                                VALUES (:1,:2,:3,:4,:5)''',rows)
        conn.commit()

In [ ]:
user = oracle_secrets['user'] + '[DVH_FAM_BB]'
dsn_tns = oracledb.makedsn(oracle_secrets['host'], 1521, service_name = oracle_secrets['service'])

with oracledb.connect(user=user, password = oracle_secrets['password'], dsn=dsn_tns) as conn:
    with conn.cursor() as cursor:
        rows = [tuple(x) for x in df_forskudd.values]
        cursor.executemany('''INSERT INTO BB_FORSKUDD_FRA_KILDE (SAKSNR, ANTALL_BARN, BELOP, AAR_MAANED) 
                                VALUES (:1,:2,:3,:4)''',rows)
        conn.commit()

In [ ]:
for filename in os.listdir('./data'):
    if os.path.isfile(os.path.join('./data', filename)):
        os.remove(os.path.join('./data', filename))